In [1]:
import pandas as pd

df = pd.read_csv('usarrests.csv')
df = df.rename(columns={df.columns[0]: 'State'})
print(df.describe())

         Murder     Assault   UrbanPop
count  50.00000   49.000000   50.00000
mean    7.78800  182.183673   74.20000
std     4.35551  130.877435   73.40828
min     0.80000   45.000000    6.00000
25%     4.07500  109.000000   53.25000
50%     7.25000  159.000000   66.00000
75%    11.25000  249.000000   77.75000
max    17.40000  879.000000  570.00000


In [2]:
df.describe()

,Murder,Assault,UrbanPop
count,50.00000,49.000000,50.00000
mean,7.78800,182.183673,74.20000
std,4.35551,130.877435,73.40828
min,0.80000,45.000000,6.00000
25%,4.07500,109.000000,53.25000
50%,7.25000,159.000000,66.00000
75%,11.25000,249.000000,77.75000
max,17.40000,879.000000,570.00000


# Immediate observations

The max UrbanPop(570) and Assault count(49 vs 50 for murder and urbanpop) clearly have issues. The UrbanPop min and Assault max look a little extreme.

In [3]:
flag_urbanpop_range = df[(df.UrbanPop > 100) | (df.UrbanPop < 0)]
flag_urbanpop_range

,State,Murder,Assault,UrbanPop
14,Iowa,2.2,56.0,570


# flag 1
The UrbanPop value cannot be more than 100 because it is a percentage of the states population. An Urbanpop of 570 is impossible.
Action: Correct the value based on the original source or set the value to NaN if that's not possible. It looks like it could have been a decimal typo, so it could have been 57.0.

In [4]:
df = df.drop(flag_urbanpop_range.index)
df = df.reset_index(drop=True)

In [5]:
df.describe()

,Murder,Assault,UrbanPop
count,49.000000,48.000000,49.000000
mean,7.902041,184.812500,64.081633
std,4.324566,130.948529,16.592966
min,0.800000,45.000000,6.000000
25%,4.300000,109.000000,53.000000
50%,7.300000,159.000000,66.000000
75%,11.300000,249.750000,77.000000
max,17.400000,879.000000,91.000000


In [6]:
flag_urbanpop_low = df.sort_values('UrbanPop').head(5)
flag_urbanpop_low

,State,Murder,Assault,UrbanPop
30,New York,11.1,254.0,6
43,Vermont,2.2,48.0,32
46,West Virginia,5.7,81.0,39
32,North Dakota,0.8,45.0,44
22,Mississippi,16.1,259.0,44


# flag 2
New York having an UrbanPop of 6 is extreme and probably not accurate. Logically it is likely incorrect, but it is technically a possible number.
Action: Compare it to the original source to see if it is an error, but don't automatically drop it because it could technically be possible.

In [7]:
flag_missing = df[df.isna().any(axis=1)]
flag_missing

,State,Murder,Assault,UrbanPop
9,Georgia,17.4,NaN,60


# flag 3
Georgia is missing a recorded value for assault. This does not seem to be significantly skewing any of the summary table values.
Action: Leave it as NaN.

In [8]:
flag_assault_outlier = df.sort_values('Assault', ascending=False).head(5)
flag_assault_outlier

,State,Murder,Assault,UrbanPop
38,South Carolina,14.4,879.0,48
31,North Carolina,13.0,337.0,45
8,Florida,15.4,335.0,80
18,Maryland,11.3,300.0,67
2,Arizona,8.1,294.0,80


# flag 4
South Carolina's assault value is extreme. Assault's mean is ~185 with a std of ~131, so 879 is over 5 standard deviations out and nearly 2.6× the next-highest state. It is very high, but it is possible.
Action: Check the original source, but otherwise leave it. 